In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
!unzip "Training Data.csv.zip"


Archive:  Training Data.csv.zip
  inflating: Training Data.csv       


In [3]:
df=pd.read_csv('/content/Training Data.csv')

In [4]:
df.head()

,Id,Income,Age,Experience,Married/Single,House_Ownership,Car_Ownership,Profession,CITY,STATE,CURRENT_JOB_YRS,CURRENT_HOUSE_YRS,Risk_Flag
0,1,1303834,23,3,single,rented,no,Mechanical_engineer,Rewa,Madhya_Pradesh,3,13,0
1,2,7574516,40,10,single,rented,no,Software_Developer,Parbhani,Maharashtra,9,13,0
2,3,3991815,66,4,married,rented,no,Technical_writer,Alappuzha,Kerala,4,10,0
3,4,6256451,41,2,single,rented,yes,Software_Developer,Bhubaneswar,Odisha,2,12,1
4,5,5768871,47,11,single,rented,no,Civil_servant,Tiruchirappalli[10],Tamil_Nadu,3,14,1


In [5]:
df.shape

(252000, 13)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252000 entries, 0 to 251999
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Id                 252000 non-null  int64 
 1   Income             252000 non-null  int64 
 2   Age                252000 non-null  int64 
 3   Experience         252000 non-null  int64 
 4   Married/Single     252000 non-null  object
 5   House_Ownership    252000 non-null  object
 6   Car_Ownership      252000 non-null  object
 7   Profession         252000 non-null  object
 8   CITY               252000 non-null  object
 9   STATE              252000 non-null  object
 10  CURRENT_JOB_YRS    252000 non-null  int64 
 11  CURRENT_HOUSE_YRS  252000 non-null  int64 
 12  Risk_Flag          252000 non-null  int64 
dtypes: int64(7), object(6)
memory usage: 25.0+ MB


In [7]:
df.isnull().sum()

,0
Id,0
Income,0
Age,0
Experience,0
Married/Single,0
House_Ownership,0
Car_Ownership,0
Profession,0
CITY,0
STATE,0


In [8]:
!pip install category-encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 1.8 MB/s eta 0:00:00


In [9]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import category_encoders as ce
from sklearn import preprocessing


In [10]:
label_encoder = preprocessing.LabelEncoder()

In [11]:

label_encoder = LabelEncoder()

for col in ['Married/Single','Car_Ownership']:
    df[col] = label_encoder.fit_transform( df[col] )

In [12]:

pd.get_dummies(df, columns=["House_Ownership"])

,Id,Income,Age,Experience,Married/Single,Car_Ownership,Profession,CITY,STATE,CURRENT_JOB_YRS,CURRENT_HOUSE_YRS,Risk_Flag,House_Ownership_norent_noown,House_Ownership_owned,House_Ownership_rented
0,1,1303834,23,3,1,0,Mechanical_engineer,Rewa,Madhya_Pradesh,3,13,0,False,False,True
1,2,7574516,40,10,1,0,Software_Developer,Parbhani,Maharashtra,9,13,0,False,False,True
2,3,3991815,66,4,0,0,Technical_writer,Alappuzha,Kerala,4,10,0,False,False,True
3,4,6256451,41,2,1,1,Software_Developer,Bhubaneswar,Odisha,2,12,1,False,False,True
4,5,5768871,47,11,1,0,Civil_servant,Tiruchirappalli[10],Tamil_Nadu,3,14,1,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251995,251996,8154883,43,13,1,0,Surgeon,Kolkata,West_Bengal,6,11,0,False,False,True
251996,251997,2843572,26,10,1,0,Army_officer,Rewa,Madhya_Pradesh,6,11,0,False,False,True
251997,251998,4522448,46,7,1,0,Design_Engineer,Kalyan-Dombivli,Maharashtra,7,12,0,False,False,True
251998,251999,6507128,45,0,1,0,Graphic_Designer,Pondicherry,Puducherry,0,10,0,False,False,True


In [13]:
onehot_encoder = OneHotEncoder(sparse_output = False)
df['House_Ownership'] = onehot_encoder.fit_transform(df['House_Ownership'].values.reshape(-1, 1) )


In [14]:
high_card_features = ['Profession', 'CITY', 'STATE']

count_encoder = ce.CountEncoder()


count_encoded = count_encoder.fit_transform( df[high_card_features] )
df = df.join(count_encoded.add_suffix("_count"))

In [15]:

df=df.drop(labels=['Profession', 'CITY', 'STATE'], axis=1)

In [16]:
df.head()

,Id,Income,Age,Experience,Married/Single,House_Ownership,Car_Ownership,CURRENT_JOB_YRS,CURRENT_HOUSE_YRS,Risk_Flag,Profession_count,CITY_count,STATE_count
0,1,1303834,23,3,1,0.0,0,3,13,0,5217,798,14122
1,2,7574516,40,10,1,0.0,0,9,13,0,5053,849,25562
2,3,3991815,66,4,0,0.0,0,4,10,0,5195,688,5805
3,4,6256451,41,2,1,0.0,1,2,12,1,5053,607,4658
4,5,5768871,47,11,1,0.0,0,3,14,1,4413,809,16537


In [17]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

In [21]:
import numpy as np

In [18]:
X = df.drop('Risk_Flag', axis=1)
y = df['Risk_Flag']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Laplace Smoothing

In [20]:
pipeline = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ('mnb', MultinomialNB())
])

In [22]:
param_dist = {
    'mnb__alpha': np.linspace(0.1, 2.0, 20)
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=10,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1
)

In [23]:
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('smote', SMOTE(random_state=42)),
                                             ('mnb', MultinomialNB())]),
                   n_jobs=-1,
                   param_distributions={'mnb__alpha': array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. , 1.1, 1.2, 1.3,
       1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2. ])},
                   random_state=42, scoring='f1')

In [24]:
print("Best alpha (Laplace smoothing):", random_search.best_params_)

Best alpha (Laplace smoothing): {'mnb__alpha': np.float64(0.1)}


In [25]:
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.516984126984127
              precision    recall  f1-score   support

           0       0.88      0.52      0.65     44201
           1       0.13      0.52      0.21      6199

    accuracy                           0.52     50400
   macro avg       0.51      0.52      0.43     50400
weighted avg       0.79      0.52      0.60     50400

